# ATTIRE CHECKER AND FACE RECOGNITION SYSTEM 

> This system is designed to check the attire of student in TARUMT and to detect faces of student for face recognition.

![FR_gif](https://i.pinimg.com/originals/2e/16/a8/2e16a81155d82049a213a3bc0c7ce1bc.gif)

## This code is used to install library.

In [7]:
# install dependencies:-
#!pip uninstall opencv-python
#!pip install opencv-contrib-python
#!pip install Pillow
# !pip install --upgrade google-cloud-storage
# pip install --upgrade firebase-admin
#pip3 install pycryptodome
#!pip install opencv-contrib-python --upgrade  if you are faci9ng module import errors!!.
#pip install yagmail 

## This code is allow to run if only you wanted to create a new names list for dataset.

In [8]:
 # creating pickle file for dataset list.
# '''DO NOT RUN THESE LINES TWICE'''
# import pickle

# names = []
# filename = "names.pkl"
# f = open(filename, 'wb')
# pickle.dump(names,f)
# f.close()

## Dependencies (library import):-
1. os :- for navigation through directories and file saving purposes\
2. cv2 :- for computer vision and machine learning
3. numpy:- to handle encoded image data inform of arrays
4. PIL.image :- handling images for opening, manipulating, saving
5. pickle :- file handling for saving names corresponding to face
6. tkinter :- to show user interface and handling input from UI
7. partial :- derive parameters from one function to another function
8. pyrebase :- allow to upload picture to cloud storage
9. firebase_admin :- allow updating data to firestore

# Face Registration
---
## Face Sampling

Used __Haarcascade__ for detecting facial features and detecting a face. As Its is efficient and fast method with high accuracy.
The algorithm used in Local Binary Patterns Histograms (LBPH). 

Face Detection: it has the objective of finding the faces (location and size) in an image and probably extract them to be used by the face recognition algorithm.

Face Recognition: with the facial images already extracted, cropped, resized and usually converted to grayscale, the face recognition algorithm is responsible for finding characteristics which best describe the image.

## Face Training

Used LBPH Algorithm from opencv library for training and modelling purpose to avoid illumination differences in image.





In [ ]:
import cv2
import numpy as np
from tkinter import Tk, Button, Canvas, messagebox, Label, StringVar, Entry, BOTH, END, LEFT,RIGHT, ttk, Frame, Toplevel, PhotoImage
from tkinter import filedialog,Menu,OptionMenu
from functools import partial
from PIL import Image, ImageFont, ImageTk
import pickle
import os
import pyrebase
from datetime import datetime
from datetime import date
import re
from tkcalendar import DateEntry
import csv
import yagmail
from firebase_dbDecouple import firedb
from MainSound3 import MainPage
from detect_cleaning import detect
from threading import *


user = yagmail.SMTP(user='calebckl-wp19@student.tarc.edu.my', \
                       password='990623127071')


face_detector = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')


#popup windows for report_window
def report_window(admin):
    report_window = Toplevel(admin)
    report_window.geometry('500x350+550+400')
    report_window.resizable(False,False)
    report_window.title("Generate Report") 

    global report_butt_green
    report_butt_green = PhotoImage(file="images/report_butt_green.png")

    global cancel_butt
    cancel_butt = PhotoImage(file= "images/cancel_butt.png")

    #grab_events(report_window)
    report_window.attributes('-topmost',True)

    Label(report_window, text= "Please pick the time range to generate report", pady=25, font= ("Helvetica", 14),wraplength=310).pack()

    labelFrame = Frame(report_window)

    Label(labelFrame, text="Start Date: ").pack(side=LEFT)
    Label(labelFrame, text="End Date: ").pack(padx=(170,0))

    labelFrame.pack( padx=70)



    calFrame= Frame(report_window)
    startDate = StringVar()
    #used to trace the startDate to serve as baseline for the endDate
    def updateDate(*args):
        dt = startDate.get()
        if (len(dt)>5):
            dt1 = datetime.strptime(dt, '%d-%m-%Y')
            endCal.config(mindate = dt1)
    #start date entry
    initDate = date(2023,3,14)
    lastDate = date.today() #today date as max date
    startCal = DateEntry(calFrame,textvariable =startDate, date_pattern='dd-mm-yyyy', locale = 'en_US',selectmode='day',mindate= initDate, maxdate= lastDate, year=2022, month =3, day =14)
    startCal.pack(side= LEFT)

    #trave what user has selected at the startDate calendar
    startDate.trace('w', updateDate)

    #end date entry
    endCal = DateEntry(calFrame,locale = 'en_US', date_pattern='dd-mm-yyyy',selectmode='day',mindate= initDate, maxdate= lastDate)
    endCal.pack(padx=(130,0))

    calFrame.pack( padx=70)

    falFrame = Frame(report_window)
    Label(falFrame, text="Faculty Name: ").pack(side=LEFT)
    
    options = [
    "FOCS",
    "FAFB",
    "FOET",
    "FSSH",
    "FCCI",
    "FOBE",
    "FOAS"
    ]

    # datatype of menu text
    clicked = StringVar()

    # initial menu text
    clicked.set( "FOCS" )

    # Create Dropdown menu
    drop = OptionMenu(falFrame , clicked , *options )
    drop.pack(padx=(100,0))
    faculty = clicked.get()
    
    
    falFrame.pack( padx=30,pady=20)
    
    
    buttonFrame = Frame(report_window)
    button_report_green = Button(buttonFrame, image= report_butt_green, borderwidth=0, command=lambda:save_file(report_window))
    button_report_green.pack(side=LEFT,padx=(50,18))
    Button(buttonFrame, image= cancel_butt, borderwidth=0, command= lambda: report_window.destroy()).pack(side=RIGHT,padx=(18,50))

    def save_file(report_window: Toplevel):
        startingDate =startCal.get_date()
        endingDate = endCal.get_date()
        file = filedialog.asksaveasfilename(
            filetypes=[("csv file", ".csv")],
        defaultextension=".csv",
        initialfile=f"Report_{startingDate}_{endingDate}_{faculty}.csv")
        if file:  # user selected file
            csvfile=open(file,'w', newline='')
            fieldnames = ['Blocked ID', 'Time Stamp', 'Student Name', 'Student ID','Guard ID']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            # Write header row
            writer.writeheader()
            #records = db.collection('blocked').where('date','>=',str(startingDate)).where('date','<=',str(endingDate)).stream()
            records = firedb.blocked.where('date','>=',str(startingDate)).where('date','<=',str(endingDate)).stream()
            #add faculty
            for record in records:
                data = record.to_dict()
                studentID = data['studentID']
                student_ref = firedb.student
                studentref = student_ref.stream() #access the each docs in the collection "Users"
        
                for doc in studentref:
                    if doc.id == studentID:
                        if doc.to_dict()['faculty'] == faculty:
                            writer.writerow({
                                'Blocked ID': data['blockedID'],
                                'Time Stamp': data['date']+ " " + data['time'],
                                'Student Nameai'l
            csvfile.close()
            messagebox.showinfo("Report saved",f"The report is saved at {file}")
            user.send(to='leeks-wp19@student.tarc.edu.my', subject='Blocked Student Report', \
            contents='Dear ' + faculty + ',\n\n  This is the latest report of Blocked Student List for ' + faculty +' from ' + str(startingDate) + ' to '+ str(endingDate) +'.\n\n\n**This is an auto generate email, please do not reply.', \
            attachments=file)

        else: # user cancel the file browser window
            messagebox.showerror("Operation cancelled","No report is saved\n\nSolution: Click the 'Save' button just now or select an existing file to overwrite")
    buttonFrame.pack(pady=(80,0))

#change here
#####################################################################################################################################

def faceLearning():
    # Path for face image database
    path = 'dataset'

    recognizer = cv2.face.LBPHFaceRecognizer_create() 

    # function to get the images and label data
    def getImagesAndLabels(path):

        imagePaths = [os.path.join(path,f) for f in os.listdir(path)]     
        faceSamples=[]
        ids = []

        for imagePath in imagePaths:

            PIL_img = Image.open(imagePath).convert('L') # convert it to grayscale
            img_numpy = np.array(PIL_img,'uint8')

            id = int(os.path.split(imagePath)[-1].split(".")[1])
            faces = face_detector.detectMultiScale(img_numpy)

            for (x,y,w,h) in faces:
                faceSamples.append(img_numpy[y:y+h,x:x+w])
                ids.append(id)

        return faceSamples,ids

    print ("\nTraining for the faces has been started. It might take a while.\n")
    faces,ids = getImagesAndLabels(path)
    recognizer.train(faces, np.array(ids))

    # Save the model into trainer/trainer.yml
    recognizer.write('trainer/trainer.yml') 

    # Print the numer of faces trained and end program
    print("{0} faces trained. Exiting Training Program".format(len(np.unique(ids))))   
    messagebox.showinfo("Done", "Faces has been trained! Click <Ok> to Go Back!")

def faceUpload(choose):
     #Create an instance of Tkinter frame or window
    faceUpload = Frame(window)

    #Set the geometry of tkinter frame
    faceUpload.pack(fill='both', expand=True)
    choose.pack_forget()
    faceUpload.configure (bg='white')


    with open('names.pkl', 'rb') as f:
        names = pickle.load(f)

    def verify():
        size = len(names)
        x = 0
        name = datanameEntry.get()
        T = True
        a = False
        results = firedb.student.where("studentID", "==", name).get()
        for result in results:
            a = True

        if name == '':
            window.attributes('-topmost',False)
            messagebox.showerror("Try Again", "Your input is empty!")
            window.attributes('-topmost',True)
            datanameEntry.focus_set()
            datanameEntry.delete(0, END)
            T = False
        elif a == False:
            window.attributes('-topmost',False)
            messagebox.showerror("Try Again", "The id you entered is not existed!")
            window.attributes('-topmost',True)
            datanameEntry.focus_set()
            datanameEntry.delete(0, END)
            T = False
        else:
            while x < size:
                if name == names[x]:
                    T = False
                    window.attributes('-topmost',False)
                    messagebox.showerror("Try Again", "This id you entered is registered!")
                    window.attributes('-topmost',True)
                    datanameEntry.focus_set()
                    datanameEntry.delete(0, END)
                    x += 1
                else:
                    x += 1

        if T:

            t2 = Thread(target=output)    
            t2.start()

            faceUpload.pack_forget()
            admin()

    def output():

        name = datanameEntry.get()
        names.append(name)
        id = names.index(name) 

        # Initialize individual sampling face count
        count = 0

        path = "face"
        images_paths = [os.path.join(path,f) for f in os.listdir(path)]
        
        for image in images_paths:
            image = cv2.imread(image)
            gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale( 
            gray,
            scaleFactor = 1.2,
            minNeighbors = 5,
            #minSize = (int(minW), int(minH)),
           )
            #if len(faces) =! 0:
            if faces is not (): 
                for (x,y,w,h) in faces:
                    while count < 50:
                        count += 1
                        cv2.imwrite("dataset/"+name+"." + str(id) + '.' + str(count) + ".jpg", gray[y:y+h,x:x+w])

                with open('names.pkl', 'wb') as f:
                    pickle.dump(names, f)
        
                messagebox.showwarning("Warning", "Training is started! Please wait.")
                faceLearning()
            
                

            else:
                messagebox.showerror("Try Again","Faces is not found from gallery!")
                datanameEntry.delete(0, END)


    def back():
        choosing(faceUpload)

     #TARUMT LOGO
    img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
    label = Label(faceUpload,image = img)
    label.image = img
    label.grid(row = 0, columnspan = 2,padx = (100, 10), pady = (50,10))
    label.grid_rowconfigure(1, weight=1)
    label.grid_columnconfigure(1, weight=1)
    label.configure(bg='white')

    welcomeLabel = Label(faceUpload, text = "Enter a valid student ID to register!",font = ('Century 20 bold'))
    welcomeLabel.grid(row = 1, columnspan = 2,padx = (100, 10), pady = (20,10)) 
    welcomeLabel.configure(bg='white')

    datanameLabel = Label(faceUpload, text = "Enter Student ID:",font = ('Century 20 bold'),bg=('white')).grid(row = 2, column = 0,padx = (10, 10),pady = (10,10))
    datanameEntry = Entry(faceUpload,font = ('Century 20 bold'),bg=('#EEE4E4'))
    datanameEntry.focus_set()
    datanameEntry.grid(row = 2, column = 1,padx = (50, 10), pady = (10,10)) 

    btn1 = Button(faceUpload, text="Enter", command= verify,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn1.grid(row = 3, column = 1,padx = (100, 10), pady = (10,10)) 

    window.bind('<Return>', lambda event:verify( ))

    btn2 = Button(faceUpload, text="Back", command= back,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn2.grid(row = 3, column = 0,padx = (100, 10), pady = (10,10)) 

def faceSampling(choose):
    #Create an instance of Tkinter frame or window
    faceSampling = Frame(window)
    #Set the geometry of tkinter frame
    faceSampling.pack(fill='both', expand=True)
    choose.pack_forget()
    faceSampling.configure (bg='white')

    face_detector = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
    # describe the type of font to be used.
    font = cv2.FONT_HERSHEY_SIMPLEX

    with open('names.pkl', 'rb') as f:
        names = pickle.load(f)

    def verify():
        size = len(names)
        x = 0
        name = datanameEntry.get()
        T = True
        a = False
        results = firedb.student.where("studentID", "==", name).get()
        for result in results:
            a = True

        if name == '':
            window.attributes('-topmost',False)
            messagebox.showerror("Try Again", "Your input is empty!")
            window.attributes('-topmost',True)
            datanameEntry.focus_set()
            datanameEntry.delete(0, END)
            T = False
        elif a == False:
            window.attributes('-topmost',False)
            messagebox.showerror("Try Again", "The id you entered is not existed!")
            window.attributes('-topmost',True)
            datanameEntry.focus_set()
            datanameEntry.delete(0, END)
            T = False
        else:
            while x < size:
                if name == names[x]:
                    T = False
                    window.attributes('-topmost',False)
                    messagebox.showerror("Try Again", "This id you entered is registered!")
                    window.attributes('-topmost',True)
                    datanameEntry.focus_set()
                    datanameEntry.delete(0, END)
                    x += 1
                else:
                    x += 1

        if T:

            t2 = Thread(target=output)    
            t2.start()

            faceSampling.pack_forget()
            admin()
            #output()

    def output():

        cam = cv2.VideoCapture(0)
        cam.set(3, 1280) # set video width
        cam.set(4, 720) # set video height

        name = datanameEntry.get()
        names.append(name)
        id = names.index(name) 


        # Initialize individual sampling face count
        count = 0

        while True:
            ret, img = cam.read()
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale( 
            img,
            scaleFactor = 1.3,
            minNeighbors = 5)
            #inserting text on video
            cv2.putText(img, 
                        'Move face and change expression', 
                        (50, 50), 
                        font, 1, 
                        (255, 0, 0), 
                        3, 
                        cv2.LINE_AA)

            #if faces != ():
            for (x,y,w,h) in faces:

                cv2.rectangle(img, (x,y), (x+w,y+h), (0,255,0), 1)   
                cv2.putText(img,str(count),(10,90),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
                count += 1

                # Save the captured image into the datasets folder
                cv2.imwrite("dataset/"+name+"." + str(id) + '.' + str(count) + ".jpg", gray[y:y+h,x:x+w])

            # Display the resulting frame
            cv2.imshow('image', img)

            #from IPython.display import Audio
            #sound_file = 'C:/Users/caleb/Downloads/Face-Recognition-LBPH-master/voice.mp3'

            #if count == 10:
            #Audio(sound_file, autoplay=True)

            k = cv2.waitKey(100) & 0xff # Press 'ESC' for exiting video
            if k == 27:
                cv2.destroyAllWindows()
            elif cv2.getWindowProperty('image',cv2.WND_PROP_VISIBLE) < 1:        
                break
            elif count >= 300: # Take 130 face sample and stop video
                 break

        with open('names.pkl', 'wb') as f:
            pickle.dump(names, f)

        cam.release()
        cv2.destroyAllWindows()
        messagebox.showwarning("Warning", "Training is started! Please wait.")
        faceLearning()

    def back():
        choosing(faceSampling)

    #TARUMT LOGO
    img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
    label = Label(faceSampling,image = img)
    label.image = img
    label.grid(row = 0, columnspan = 2,padx = (100, 10), pady = (50,10))
    label.grid_rowconfigure(1, weight=1)
    label.grid_columnconfigure(1, weight=1)
    label.configure(bg='white')

    welcomeLabel = Label(faceSampling, text = "Enter a valid student ID to register!",font = ('Century 20 bold'))
    welcomeLabel.grid(row = 1, columnspan = 2,padx = (100, 10), pady = (20,10)) 
    welcomeLabel.configure(bg='white')

    datanameLabel = Label(faceSampling, text = "Enter Student ID:",font = ('Century 20 bold'),bg=('white')).grid(row = 2, column = 0,padx = (10, 10),pady = (10,10))
    datanameEntry = Entry(faceSampling,font = ('Century 20 bold'),bg=('#EEE4E4'))
    datanameEntry.focus_set()
    datanameEntry.grid(row = 2, column = 1,padx = (50, 10), pady = (10,10)) 

    btn1 = Button(faceSampling, text="Enter", command= verify,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn1.grid(row = 3, column = 1,padx = (100, 10), pady = (10,10)) 

    btn2 = Button(faceSampling, text="Back", command= back,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn2.grid(row = 3, column = 0,padx = (100, 10), pady = (10,10)) 

    window.bind('<Return>', lambda event:verify( ))


def choosing(admin1):
    #Create an instance of Tkinter frame or window
    choose = Frame(window)
    #Set the geometry of tkinter frame
    choose.pack(fill='both', expand=True)
    admin1.pack_forget()

    #Set the geometry of tkinter frame
    choose.configure (bg='white')
    def first():
        t1 = Thread(target=faceSampling, args=(choose,))
                    
        t1.start()
        #faceSampling(choose)

    def second():
        faceUpload(choose)

    def off():
        choose.pack_forget()
        admin()
        

    img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
    label = Label(choose,image = img)
    label.image = img
    label.pack(padx = (30, 30), pady = (50,10))
    label.configure(bg='white')

    welcomeLabel = Label(choose, text = "Click the button to start!",font = ('Century 20 bold'))
    welcomeLabel.pack(padx = (30, 30), pady = (20,10)) 
    welcomeLabel.configure(bg='white')

    btn1 = Button(choose, text="Register Face With Camera", command= first,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn1.pack(padx = 30,pady=30) 

    btn2 = Button(choose, text="Register Face with Gallery", command= second,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn2.pack(padx = 30,pady=30)

    btn3 = Button(choose, text="Back", command= off,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn3.pack(padx = 30,pady=30)

def security(log):
    #Create an instance of Tkinter frame or window
    security = Frame(window)
    security.pack(fill='both', expand=True)

    #Set the geometry of tkinter frame
    security.configure (bg='white')

    login = log
    def first1() :
        # faceRecognition(login)
        security.pack_forget()
        Main = MainPage(window)
        Main.configure(bg='white')
        Main.pack()
        
        

    def second2():
        security.pack_forget()

    def ok() :
        window.destroy()

    img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
    label = Label(security,image = img)
    label.image = img
    label.pack(padx = (30, 30), pady = (50,10))
    label.configure(bg='white')

    welcomeLabel = Label(security, text = "Click the button to start!",font = ('Century 20 bold'))
    welcomeLabel.pack(padx = (30, 30), pady = (20,10)) 
    welcomeLabel.configure(bg='white')

    #Create a Label and a Button widget

    btn1 = Button(security, text="Face Recognization", command= first1,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn1.pack(padx = 30,pady=30)

    btn2 = Button(security, text="Switch to Admin", command= second2,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn2.pack(padx = 30,pady=30)

    btn3 = Button(security, text="Exit", command= ok,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn3.pack(padx = 30,pady=30)

def admin():

    admin = Frame(window)
    admin.pack(fill='both', expand=True)

    #Set the geometry of tkinter frame
    admin.configure (bg='white')

    def first1() :
        choosing(admin)

    def second2():
        report_window(admin)

    def ok() :
        window.destroy()

    img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
    label = Label(admin,image = img)
    label.image = img
    label.pack(padx = (30, 30), pady = (50,10))
    label.configure(bg='white')

    welcomeLabel = Label(admin, text = "Click the button to start!",font = ('Century 20 bold'))
    welcomeLabel.pack(padx = (30, 30), pady = (20,10)) 
    welcomeLabel.configure(bg='white')


    #Create a Label and a Button widget
    btn1 = Button(admin, text="Register New Face", command= first1,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn1.pack(padx = 30,pady=30)

    btn2 = Button(admin, text="Generate Report", command= second2,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn2.pack(padx = 30,pady=30)

    btn3 = Button(admin, text="Exit", command= ok,font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
    btn3.pack(padx = 30,pady=30)


#changes here
def validateLogin(username, password, window:Tk):
    id = username.get()
    a = True
    adminpw = 'adminPassword'
    sepw ='guardPassword'
    if username.get() != "":
        result = firedb.admin.document(id).get()
        result1 = firedb.security.document(id).get()
    pw = password.get()

    if username.get() == "" or password.get() == "":
        messagebox.showerror("Try Again", "Please enter your username and password!")
        usernameEntry.focus_set()
        clear_text()
    elif result.exists:
        admin_ref = firedb.admin
        adminref = admin_ref.stream() #access the each docs in the collection "Users"
        
        for doc in adminref:
            if doc.id == id:
                if doc.to_dict()[adminpw] == pw:
                    #window.destroy()
                    usernameEntry.focus_set()
                    clear_text()
                    admin()
                else:
                    #Error message if empty input
                    messagebox.showerror("Try Again", "Please enter your username and password!")
                    usernameEntry.focus_set()
                    clear_text()
       
    elif result1.exists:
        security_ref = firedb.security
        securityref = security_ref.stream() #access the each docs in the collection "security"

        for doc in securityref:
            if doc.id == id: #id represent the guardID here
                if doc.to_dict()[sepw] == pw:
                    # login = id
                    usernameEntry.focus_set()
                    clear_text()

                    ##logined guardid (string)
                    # security(login) #passtoKS main
                    window.destroy()

                    window = Tk()
                    securityPage = MainPage(window, id)
                    securityPage.pack()
                    window.mainloop()

                   # a = False
                else:
                    #Error message if empty input
                    messagebox.showerror("Try Again", "Please enter your username and password!")
                    usernameEntry.focus_set()
                    clear_text()
                   # a = False
    else:
        messagebox.showerror("Try Again", "You have entered a wrong username or password!")
        usernameEntry.focus_set()
        clear_text()

def exit():
    window.destroy()

#window
window = Tk()  
window.geometry('750x700')  
#frame = Frame(win, width=600, height=400)
window.title('Login')
#window.attributes('-topmost',True)
window.configure (bg='white')

#TARUMT LOGO
img = ImageTk.PhotoImage(Image.open("tarumt-logo1.png"))
label = Label(image = img)
label.image = img
label.place(x=175, y=100)
label.configure(bg='white')

welcomeLabel = Label(window, text = "Welcome to Attire Checker\nand Face Recognition system!",font = ('Century 20 bold'))
welcomeLabel.place(x=175, y=240)
welcomeLabel.configure(bg='white')

#username label and text entry box
usernameLabel = Label(window, text = "Username:",font = ('Century 20 bold'))
usernameLabel.place(x=100, y=350)
usernameLabel.configure(bg='white')

username = StringVar()
usernameEntry = Entry(window,font = ('Century 20 bold'),bg=('#EEE4E4'))
usernameEntry.place(x=280, y=350)
#usernameEntry.configue(bg='DAD0D0')

#password label and password entry box
passwordLabel = Label(window,text = "Password:",font = ('Century 20 bold'))
passwordLabel.place(x=110, y=420)
passwordLabel.configure(bg='white')

password = StringVar()
passwordEntry = Entry(window, show = '*',font = ('Century 20 bold'),bg=('#EEE4E4'))
passwordEntry.place(x=280, y=420)
#passwordEntry.configue(bg='DAD0D0')

#initializing clothing detector
#clothingDetector = detect('best.pt')

validateLogin = partial(validateLogin, usernameEntry, passwordEntry, window) #changes here

def clear_text() :
    usernameEntry.delete(0, END)
    passwordEntry.delete(0, END)

#login button
Btn = Button(window, text = "Login", command = validateLogin, font = ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
Btn.place(x=450, y=510)
window.bind('<Return>', lambda event:validateLogin( ))

Btn1 = Button(window, text = "Exit", command = exit, font =  ('Century 20 bold'),bg=("#6459C1"), fg=('white'))
Btn1.place(x=170, y=510) 

window.mainloop()

<>:277: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:277: SyntaxWarning: "is not" with a literal. Did you mean "!="?
C:\Users\caleb\AppData\Local\Temp\ipykernel_18180\2773618387.py:277: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if faces is not ():


Done.
1 ripped_pants  

yes detected
activeLearning/14124812552008.png
False
Done.
1 Sleeveless  

yes detected
activeLearning/25201225311804.png
False
